# 02 State Machine, OLS Betas, And Per-Pitcher Models

This is the professor-aligned core notebook: state transitions by pitcher/batter/count/previous pitch, OLS beta tables, then the per-pitcher boosted loop compared with one pooled model.

In [ ]:
# Colab/local setup. Keep helper functions at the top of every notebook.
import os
import sys
import subprocess
from pathlib import Path

# Check if running in Google Colab environment
IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # Optional: set after the public GitHub repo exists.
TOP_N = 100 # Define a constant for the top N items, likely used for filtering data.

# Set up the base directory for the project, handling both Colab and local environments
if IN_COLAB:
    # Install required packages if in Colab. First attempt to use requirements.txt, then fall back to individual packages.
    %pip -q install -r requirements.txt || %pip -q install pybaseball pandas numpy scikit-learn matplotlib seaborn pyarrow shap statsmodels xgboost tqdm
    # Clone the repository if a REPO_URL is provided and it hasn't been cloned yet
    if REPO_URL and not Path("/content/pitch-sequencing").exists():
        !git clone {REPO_URL} /content/pitch-sequencing
    # Define BASE_DIR based on whether the repo was cloned or if in Colab (current working directory)
    BASE_DIR = Path("/content/pitch-sequencing") if Path("/content/pitch-sequencing").exists() else Path.cwd()
else:
    # If not in Colab, set BASE_DIR to the current working directory
    BASE_DIR = Path.cwd()

# Change the current working directory to BASE_DIR
os.chdir(BASE_DIR)
# Add the 'code' directory to the system path for importing local modules
sys.path.insert(0, str(BASE_DIR / "code"))

# Define paths for data, processed data, and output directories
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# Create these directories if they don't already exist
for path in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, OUTPUT_DIR / "figures"]:
    path.mkdir(parents=True, exist_ok=True)

# Helper function to run external scripts/commands
def run_step(args):
    print("Running:", " ".join(map(str, args)))
    # Execute the command and capture output
    result = subprocess.run(args, cwd=BASE_DIR, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode() # Raise an exception if the command returned a non-zero exit code

# Helper function to display diagnostic information about a DataFrame
def frame_diag(df, label):
    print(f"{label}: rows={len(df):,}, cols={df.shape[1]:,}")
    # Print unique pitcher count if 'pitcher_name' column exists
    if "pitcher_name" in df:
        print(f"{label}: pitchers={df['pitcher_name'].nunique():,}")
    # Print unique pitch type count and value counts if 'pitch_type' column exists
    if "pitch_type" in df:
        print(f"{label}: pitch_types={df['pitch_type'].nunique():,}")
        print(df["pitch_type"].value_counts().head(12))
    return df.head() # Return the first few rows of the DataFrame

# Helper function to perform a merge and display diagnostic information
def merge_diag(left, right, keys, label):
    print(f"[merge:{label}] before left_rows={len(left):,}, right_rows={len(right):,}, keys={keys}")
    print(f"[merge:{label}] right_duplicate_keys={right.duplicated(keys).sum():,}")
    # Perform a left merge, validating as many-to-one and adding an indicator column
    merged = left.merge(right, on=keys, how="left", validate="many_to_one", indicator=True)
    # Print post-merge statistics, including unmatched rows
    print(f"[merge:{label}] after rows={len(merged):,}, unmatched={merged['_merge'].eq('left_only').sum():,}")
    return merged.drop(columns=["_merge"]) # Return the merged DataFrame without the indicator column

# Helper function to display the head of a CSV file
def show_csv(path, rows=8):
    import pandas as pd # Import pandas locally within the function
    path = BASE_DIR / path # Construct the full path to the CSV file
    print(path)
    df = pd.read_csv(path) # Read the CSV into a DataFrame
    display(df.head(rows)) # Display the first 'rows' of the DataFrame
    return df # Return the DataFrame

In [ ]:
# Run the 'state_machine_and_ols.py' script with TOP_N as an argument.
# This script likely performs state machine analysis and OLS regression.
run_step([sys.executable, "code/state_machine_and_ols.py", "--top-n", str(TOP_N)])

# Load and display various output CSV files generated by the script.
# These files contain model metrics, OLS beta coefficients, and state transition data.
metrics = show_csv("output/state_ols_model_comparison_metrics.csv") # Model comparison metrics
betas = show_csv("output/ols_beta_pitcher_pitch_intercepts.csv") # OLS beta intercepts for pitcher and pitch type
beta_summary = show_csv("output/ols_beta_feature_summary.csv") # Summary of OLS beta features
transitions = show_csv("output/state_machine_top3_transitions.csv") # Top 3 state machine transitions

Key outputs:
`state_machine_transitions.csv`,
`state_machine_top3_transitions.csv`,
`ols_pitch_type_coefficients_long.csv`,
`ols_beta_pitcher_pitch_intercepts.csv`,
`ols_beta_pitcher_pitch_mean_abs_context.csv`,
`state_ols_model_comparison_metrics.csv`.